In [1]:
import seaborn
from scipy.special import softmax, log_softmax
from numpy.lib.stride_tricks import sliding_window_view
import scipy.stats as stats
import pickle
import json
from pathlib import Path
import warnings
import torch
import re
from collections import defaultdict
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import scipy
import numpy as np
from seaborn import kdeplot
from tqdm.auto import tqdm
from scipy.stats import gmean
from sklearn.metrics import roc_curve, auc


def meanprob(answer_probabilities):
    return gmean([next(iter(value.values())) for value in answer_probabilities])

def sort_key(path):
    # To sort filepaths
    match = re.search(r'traj_(\d+)(?:_sample_(\d+))?', path.name)
    if match:
        traj_num = int(match.group(1))
        sample_num = int(match.group(2)) if match.group(2) is not None else -1
        return (traj_num, sample_num)
    return (float('inf'), float('inf'))

In [2]:
folder = Path("trajectories")
# LLMS = ['gpt']
# DATASETS = ['movie_recommendation', 'math_500', 'causal_inference', 'logiqa', 'hotpotqa', 'bfcl', 'cs1qa', 'codeqa']
sample_filepaths = defaultdict(lambda: defaultdict(dict))

sample_filepaths['llama']['movie_recommendation']['vanilla'] = folder/"llama-bm-723-batch_llama_bigbench_movie_sfull/vanilla"
sample_filepaths['llama']['movie_recommendation']['rejection'] = folder/"llama-bm-723-batch_llama_bigbench_movie_sfull/rejection"
sample_filepaths['llama']['movie_recommendation']['lawyer'] = folder/"llama-bm-723-batch_llama_bigbench_movie_sfull/lawyer"
sample_filepaths['llama']['movie_recommendation']['stepbootstrap'] = folder/"llama-bm-723-batch_llama_bigbench_movie_sfull/stepbootstrap"


In [3]:
data = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(list))))

for llm in sample_filepaths: 
    for dataset in sample_filepaths[llm]:
        for method in sample_filepaths[llm][dataset]:
            print(f"Loading {method} @ {dataset} @ {llm}")
            trajectories = []
            sample_files = sorted(sample_filepaths[llm][dataset][method].glob("traj_*.json"), key=sort_key)
            for sample_file in tqdm(sample_files):
                with open(sample_file, "r") as file:
                    trajectory = json.load(file)
                trajectory['filepath'] = sample_file
                trajectories += [trajectory]

            resampled_confidences = 'confidences'

            for trajectory in trajectories:
                if 'final_answer' in trajectory and trajectory['final_answer']:
                    data[llm][dataset][method]['id'] += [trajectory['id']]
                    data[llm][dataset][method]['generation_time'] += [trajectory['timings']['generation_time']]
                    data[llm][dataset][method]['confidence_time'] += [trajectory['timings']['confidence_time']]
                    data[llm][dataset][method]['accuracy'] += [trajectory['evaluation_result']['correct']]
                    data[llm][dataset][method]['ground_truth'] += [trajectory['ground_truth']]
                    data[llm][dataset][method]['final_answer'] += [trajectory['final_answer']]
                    data[llm][dataset][method]['meanprob_confidence'] += [meanprob(trajectory['confidences']['answer_probabilities'])]
                    data[llm][dataset][method]['meanent_confidence'] += [np.exp(-meanprob(trajectory['confidences']['answer_entropy']))]
                    # DEBUG
                    if np.isnan(trajectory['confidences']['indirect_probabilities']['True']):
                        data[llm][dataset][method]['indirect_confidence'] += [0.5]
                    else:
                        data[llm][dataset][method]['indirect_confidence'] += [trajectory['confidences']['indirect_probabilities']['True']]
                    data[llm][dataset][method]['verbal_confidence'] += [np.sum([int(token)*prob for token, prob 
                                                                   in trajectory['confidences']['verbconf_probabilities'].items()])/100]
                    # if np.isnan(data[llm][dataset][method]['verbal_confidence'][-1]):
                    #     print(trajectory['filepath'])
                    #     print(trajectory['confidences']['verbconf_probabilities'])
            data[llm][dataset][method]['accuracy'] = torch.BoolTensor(data[llm][dataset][method]['accuracy'])
            for key in data[llm][dataset][method]:
                if key.endswith('_confidence'): 
                    data[llm][dataset][method][key] = torch.FloatTensor(data[llm][dataset][method][key])

            # Regroup per id
            new_array = defaultdict(lambda: defaultdict(list))
            for i in range(len(data[llm][dataset][method]['id'])):
                id_ = data[llm][dataset][method]['id'][i]
                for key in data[llm][dataset][method]:
                    new_array[key][id_] += [data[llm][dataset][method][key][i]]
            for key in new_array:
                new_array[key] = [v if key!='id' and len(v)>1 else v[0] for v in new_array[key].values()]
            data[llm][dataset][method] = new_array

Loading vanilla @ movie_recommendation @ llama


  0%|          | 0/257 [00:00<?, ?it/s]

Loading rejection @ movie_recommendation @ llama


  0%|          | 0/8000 [00:00<?, ?it/s]

Loading lawyer @ movie_recommendation @ llama


  0%|          | 0/8000 [00:00<?, ?it/s]

Loading stepbootstrap @ movie_recommendation @ llama


  0%|          | 0/24300 [00:00<?, ?it/s]

In [4]:
def rescale(x):
    return torch.FloatTensor(x)
    # return torch.FloatTensor(x).logit().clip(-5,5)

measure_to_time_label = {'meanprob': 'answer_prob_time', 'meanent': 'answer_ent_time', 
                         'indirect': 'indirect_time', 'verbal': 'verbconf_time'}
batch_shared_measures = {'indirect', 'verbal'}

def aggregate_confidence_time(confidence_times, measure):
    measure_times = [timing[measure_to_time_label[measure]] for timing in confidence_times]
    # Batched indirect/verbal wall time is copied into every sample record.
    # Count it once; answer probability/entropy timings are measured per sample.
    if measure in batch_shared_measures:
        return measure_times[0]
    return np.sum(measure_times)

test_statistics = defaultdict(lambda: defaultdict(list))
accuracies = defaultdict(lambda: defaultdict(list))
timing_ratios = defaultdict(lambda: defaultdict(list))

for llm in data:
    for dataset in data[llm]:
        for question_id in range(len(data[llm][dataset]['vanilla']['id'])):
            test_statistic = defaultdict(lambda: defaultdict(float))
            timing_ratio = defaultdict(lambda: defaultdict(float))
            skip = False
            
            # Vanilla
            vanilla = data[llm][dataset]['vanilla']
            vanilla_timing = {}
            for measure in ['meanprob', 'meanent', 'indirect', 'verbal']:
                test_statistic['vanilla'][measure] = rescale(vanilla[measure+'_confidence'][question_id])
                vanilla_timing[measure] = vanilla['generation_time'][question_id] + vanilla['confidence_time'][question_id][measure_to_time_label[measure]]
                timing_ratio['vanilla'][measure] = 1.0
            if vanilla['generation_time'][question_id] > 4*np.mean([t for t in vanilla['generation_time']]):
                skip = True
                continue
            
            # Rejection
            rejections = data[llm][dataset]['rejection']
            try:
                id_ = rejections['id'].index(vanilla['id'][question_id])
            except ValueError:
                print(f"  cannot find {vanilla['id'][question_id]} for rejection, skipping")
                continue
            same_answer = (rejections['final_answer'][id_]==vanilla['final_answer'][question_id])
            for measure in ['meanprob', 'meanent', 'indirect', 'verbal']:
                test_statistic['rejection'][measure] = \
                    rescale(np.append(rejections[measure+'_confidence'][id_][same_answer],
                                       vanilla[measure+'_confidence'][question_id])).mean()
                rejection_timing = vanilla['generation_time'][question_id] + rejections['generation_time'][id_][0] + aggregate_confidence_time(rejections['confidence_time'][id_], measure)
                timing_ratio['rejection'][measure] = rejection_timing/vanilla_timing[measure]

            # Lawyer
            lawyers = data[llm][dataset]['lawyer']
            try:
                id_ = lawyers['id'].index(vanilla['id'][question_id])
            except ValueError:
                print(f"  cannot find {vanilla['id'][question_id]} for laywer, skipping")
                continue
            same_answer = (lawyers['final_answer'][id_]==vanilla['final_answer'][question_id])
            for measure in ['meanprob', 'meanent', 'indirect', 'verbal']:
                test_statistic['lawyer'][measure] = \
                    rescale(np.append(lawyers[measure+'_confidence'][id_][same_answer],
                                       vanilla[measure+'_confidence'][question_id])).mean()
                lawyer_timing = vanilla['generation_time'][question_id] + lawyers['generation_time'][id_][0] + aggregate_confidence_time(lawyers['confidence_time'][id_], measure)
                timing_ratio['lawyer'][measure] = lawyer_timing/vanilla_timing[measure]

            # StepBootstrap
            stepbootstraps = data[llm][dataset]['stepbootstrap']
            try:
                id_ = stepbootstraps['id'].index(vanilla['id'][question_id])
            except ValueError:
                print(f"  cannot find {vanilla['id'][question_id]} for stepbootstrap, skipping")
                continue
            for measure in ['meanprob', 'meanent', 'indirect', 'verbal']:
                test_statistic['stepbootstrap'][measure] = \
                    rescale(np.append(stepbootstraps[measure+'_confidence'][id_],
                                       vanilla[measure+'_confidence'][question_id])).mean()
                if torch.isnan(test_statistic['stepbootstrap'][measure]):
                    skip = True
                stepbootstraps_timing = vanilla['generation_time'][question_id] + stepbootstraps['generation_time'][id_][0] + aggregate_confidence_time(stepbootstraps['confidence_time'][id_], measure)
                timing_ratio['stepbootstrap'][measure] = stepbootstraps_timing/vanilla_timing[measure]

            if not skip:
                accuracies[llm][dataset] += [vanilla['accuracy'][question_id]]
                test_statistics[llm][dataset] += [test_statistic]
                timing_ratios[llm][dataset] += [timing_ratio]

  cannot find bigbench_movie_17 for stepbootstrap, skipping
  cannot find bigbench_movie_35 for stepbootstrap, skipping
  cannot find bigbench_movie_79 for stepbootstrap, skipping
  cannot find bigbench_movie_105 for stepbootstrap, skipping
  cannot find bigbench_movie_121 for stepbootstrap, skipping
  cannot find bigbench_movie_125 for stepbootstrap, skipping
  cannot find bigbench_movie_171 for stepbootstrap, skipping


In [9]:
bootstrapped_aucs = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
bootstrapped_timeratios = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for llm in accuracies:
    for dataset in accuracies[llm]:
        methods = list(test_statistics[llm][dataset][0])
        measures = list(test_statistics[llm][dataset][0]['vanilla'])
        for _ in range(1000):
            I = np.random.choice(len(accuracies[llm][dataset]), size=len(accuracies[llm][dataset]), replace=True)
            for method in methods:
                aucs, timeratios = {}, {}
                for measure in measures:
                    fpr, tpr, _ = roc_curve(torch.stack(accuracies[llm][dataset])[I],
                                            torch.stack([t[method][measure] for t in test_statistics[llm][dataset]])[I])
                    aucs[measure] = auc(fpr, tpr)
                    timeratios[measure] = torch.FloatTensor([t[method][measure] for t in timing_ratios[llm][dataset]])[I].mean()
                bootstrapped_aucs[llm][dataset][method] += [aucs]
                bootstrapped_timeratios[llm][dataset][method] += [timeratios]

In [10]:
for llm in bootstrapped_aucs:
    print(f"******** {llm} *********")
    for dataset in bootstrapped_aucs[llm]:
        print(f"{dataset}")
        print(f"  {'':<15}  {'':<7}AUC  {'':<13}Timings")
        for method in ['vanilla', 'stepbootstrap', 'rejection', 'lawyer']:
            # bootstrapped_aucs_avg = [np.mean(list(B.values())) for B in bootstrapped_aucs[llm][dataset][method]]
            # bootstrapped_aucs_avg = [B['meanprob'] for B in bootstrapped_aucs[llm][dataset][method]]
            bootstrapped_aucs_avg = [np.mean([B[key] for key in ['meanprob', 'meanent', 'verbal']]) 
                                     for B in bootstrapped_aucs[llm][dataset][method]]
            bootstrapped_timeratios_avg = [np.mean(list(B.values())) for B in bootstrapped_timeratios[llm][dataset][method]]
            print(f"  {method: <15}: {np.mean(bootstrapped_aucs_avg): >7.4f} += {np.std(bootstrapped_aucs_avg):.4f} ", end="")
            print(f"  {np.mean(bootstrapped_timeratios_avg): >7.4f} += {np.std(bootstrapped_timeratios_avg):.4f} ")

******** llama *********
movie_recommendation
                          AUC               Timings
  vanilla        :  0.5675 += 0.0258    1.0000 += 0.0000 
  stepbootstrap  :  0.5732 += 0.0267    1.2323 += 0.0045 
  rejection      :  0.5863 += 0.0267    2.4810 += 0.0216 
  lawyer         :  0.5844 += 0.0257    2.3375 += 0.0131 


In [15]:
for llm in bootstrapped_aucs:
    print(f"******** {llm} *********")
    for dataset in bootstrapped_aucs[llm]:
        print(f"{dataset}")
        print(f"  {'':<15}  {'':<7}AUC  {'':<13}Timings")
        for method in ['vanilla', 'stepbootstrap', 'rejection', 'lawyer']:
            # bootstrapped_aucs_avg = [np.mean(list(B.values())) for B in bootstrapped_aucs[llm][dataset][method]]
            # bootstrapped_aucs_avg = [B['meanprob'] for B in bootstrapped_aucs[llm][dataset][method]]
            bootstrapped_aucs_avg = [np.mean([B[key] for key in ['meanprob']]) 
                                     for B in bootstrapped_aucs[llm][dataset][method]]
            bootstrapped_timeratios_avg = [
                B['meanprob']
                for B in bootstrapped_timeratios[llm][dataset][method]
            ]
            print(f"  {method: <15}: {np.mean(bootstrapped_aucs_avg): >7.4f} += {np.std(bootstrapped_aucs_avg):.4f} ", end="")
            print(f"  {np.mean(bootstrapped_timeratios_avg): >7.4f} += {np.std(bootstrapped_timeratios_avg):.4f} ")

******** llama *********
movie_recommendation
                          AUC               Timings
  vanilla        :  0.4591 += 0.0373    1.0000 += 0.0000 
  stepbootstrap  :  0.4647 += 0.0364    1.1853 += 0.0045 
  rejection      :  0.4925 += 0.0378    2.4645 += 0.0212 
  lawyer         :  0.4625 += 0.0372    2.3145 += 0.0129 
